# From Excel to pandas: Working with Trade Tables

We will repeat the three tasks we have just seen in Excel:

1. Add a column containing the same value in every row.
2. Label exports as **low** or **high** using a rule.
3. Calculate the mean export value for each country.

Then we will save our result as a new Excel file.

## Why use Python if we already have Excel?

Excel is useful for looking at a table, editing cells, and exploring a small dataset visually. Python is useful when we want to repeat the same steps across several files and keep a record of how we produced the result.

| Working manually in Excel | Working with Python and pandas |
|---|---|
| Repeating edits across many files can become cumbersome. | We can reuse the same instructions for several files. |
| Copying formulas or editing cells can introduce mistakes. | We can inspect a written rule and apply it consistently. Incorrect code can still produce mistakes. |
| Saving edits to the original workbook changes that file. | We can read the original and save our result in a separate folder. |
| Individual manual edits can be difficult to reconstruct without a recorded workflow or version history. | The notebook records the instructions we ran. |

Python also has a learning curve: we need to write instructions and check their results. Excel can automate work too, for example with Power Query. Today we are comparing the manual workflow we just tried with a short pandas workflow.

> **Note**
>
> All figures in these practice files are **invented for teaching**.

## 1. Open our tools

**pandas** is a Python library for working with tables. A table in pandas is called a **DataFrame**.

We will call our table **`my_df`**. This is simply the name we choose for it.

`pd` is a short name for pandas. `Path` helps us give directions to our file, just as in the previous notebook.

In [ ]:
from pathlib import Path
import pandas as pd

First check where Python is working. We expect the folder **`Lecture_4`**, where this notebook is saved.

In [ ]:
Path.cwd()

## 2. Read our first Excel table

Inside `Lecture_4`, the `tables` folder contains four Excel files: `trade_2022.xlsx`, `trade_2023.xlsx`, `trade_2024.xlsx`, and `trade_2025.xlsx`.

We will start with **`trade_2022.xlsx`**, the table used for our Excel demonstration. Each row records one country's exports to one partner in that year. Export values are expressed in **millions of US dollars**.

First give Python the relative path to the file:

In [ ]:
input_path = Path("tables") / "trade_2022.xlsx"

Now read the Excel file and store its table in `my_df`. `pd.read_excel()` reads the first worksheet by default. Reading it does not change the source file.

In [ ]:
my_df = pd.read_excel(input_path)

## 3. Look at our table

Write the table's name as the last line of a cell to display it in the notebook.

In [ ]:
my_df

**Explore together:** find Switzerland's two rows. What are its export values? Which row has exports of exactly 500?

Now look at the **row labels**, called the index:

In [ ]:
my_df.index

Our six rows have labels from `0` to `5`. These labels were created by pandas; they are not an extra column in the Excel file.

Next look at the **column names**:

In [ ]:
my_df.columns

We can change row labels and column names in a later lesson. For now, use the column names exactly as they appear, including the spelling **`export_milions_usd`**.

We can also select one column by putting its name inside square brackets:

In [ ]:
my_df["export_milions_usd"]

## 4. Add the same value to every row

In Excel, we added a column and filled it down. In pandas, one instruction applies the value to the whole column.

Create a column called **`trade_flow`** and give every row the value **`"exports"`**:

In [ ]:
my_df["trade_flow"] = "exports"

In [ ]:
my_df

**Checkpoint:** the new column contains `exports` in every row. The original columns are still present.

## 5. Add a category using a rule

Our rule is:

- Exports **greater than 500** are labelled **`"high"`**.
- Exports **500 or below** are labelled **`"low"`**.

**Predict:** which category should the row with exactly 500 receive?

First check the rule without changing the table:

In [ ]:
my_df["export_milions_usd"] > 500

Each `True` means that the row meets our rule. Each `False` means that it does not.

We will label the rows using two rules. First, assign `"low"` to rows where exports are **500 or below**:

In [ ]:
my_df.loc[ my_df["export_milions_usd"] <= 500, "export_category"] = "low"

In [ ]:
my_df

Then assign `"high"` to rows where exports are **greater than 500**:

In [ ]:
my_df.loc[my_df["export_milions_usd"] > 500, "export_category"] = "high"

Read the instruction as: **“For rows meeting this condition, put `high` in the `export_category` column.”** Inside `.loc[...]`, the part before the comma selects the rows; the part after the comma names the column to update.

In [ ]:
my_df

**Checkpoint:** 400 and 500 are labelled `low`; 800 and 900 are labelled `high`. Our practice column has no missing values, so every row can be classified using this rule.

> **Note**
>
> The `= "high"` assigns a value directly to the selected cells in `my_df`. This updates our table in Python's memory, so we do not need to write `my_df = ...` again. The original Excel file remains unchanged.

## 6. Calculate the mean for each country

Switzerland has two observations: **400** and **800**. Their mean is **600**.

Ask pandas to group rows by `country`, select the export values, and calculate their mean:

In [ ]:
country_mean = my_df.groupby("country")["export_milions_usd"].mean()
country_mean

Read it from left to right: **group by country**, take **export values**, calculate the **mean**.

**Checkpoint:** Switzerland has a mean of 600, Germany 700, and Chile 400, all in millions of US dollars. These are averages across the two partner observations for each country in 2022, not total exports.

We will explore grouping and summaries more fully in the next lesson. This result is stored separately in `country_mean`; it has not replaced `my_df`.

### What kind of object did we create?

In [ ]:
type(country_mean)

In [ ]:
pd.DataFrame(country_mean)

In [ ]:
type(pd.DataFrame(country_mean))

## 7. Save our prepared table

Keep our source files in `tables` and save our results in a separate **`outputs`** folder.

Create that output folder if it does not already exist:

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

Give the result a new filename, **`trade_2022_prepared.xlsx`**, and save it inside the `outputs` folder. This makes it easy to distinguish the prepared table from the original file.

In [ ]:
output_path = output_dir / "trade_2022_modified.xlsx"
my_df.to_excel(output_path, index=False)

`index=False` means “do not add pandas' row labels as an extra Excel column”. This saves `my_df`, including our two new columns. The separate country means remain in `country_mean`.

Open **`Lecture_4/outputs/trade_2022_modified.xlsx`** in Excel and compare it with **`Lecture_4/tables/trade_2022.xlsx`**. The prepared file contains our new columns; the original file remains unchanged.

Running the saving instruction again replaces `trade_2022_modified.xlsx` in `outputs`. Close that workbook in Excel before saving it again, particularly on Windows.

## 8. Find our Excel files

Before we work with several tables, let’s find their files.

We will look inside the `tables` folder, check which filenames end with **`.xlsx`**, and collect their paths in a list.

In [ ]:
tables_dir = Path("tables")

`excel_files` starts as an empty list. The loop below looks at each item in the folder. If its name ends with `.xlsx`, we add its path to our list.

In [ ]:
excel_files = []

for file in tables_dir.iterdir():
    if file.name.endswith(".xlsx"):
        excel_files.append(file.name)

What is tables_dir.iterdir()?

In [ ]:
list(tables_dir.iterdir())

Read the instructions from top to bottom:

- `for` visits each item in the folder.
- `if` checks whether its name ends with `.xlsx`.
- `.append(file)` adds its path to the list when the condition is met.

Now display the list:

In [ ]:
excel_files

**Checkpoint:** our list contains the filenames of the four Excel files. The CSV file is not included because its name ends with `.csv`.

This loop collects filenames only. It does not open, change, or save any tables. Run the empty-list cell again before repeating the loop so you start with a fresh list.

## What comes next in Pandas II?

We have read an Excel table, explored its rows and columns, added a fixed value and category labels, calculated country means, and saved a separate result.

In **Pandas II**, we will build on these steps and explore more ways to work with our data. This may include reading several Excel files, combining tables, and creating summaries to answer questions about trade.

## A small reference card

| What do I want to do? | Command |
|---|---|
| Import pandas | `import pandas as pd` |
| Import Path | `from pathlib import Path` |
| Check the working directory | `Path.cwd()` |
| Read an Excel file | `my_df = pd.read_excel(input_path)` |
| Display the table | `my_df` |
| Show row labels | `my_df.index` |
| Show column names | `my_df.columns` |
| Select one column | `my_df["country"]` |
| Add a fixed value to every row | `my_df["trade_flow"] = "exports"` |
| Label exports of 500 or below | `my_df.loc[my_df["export_milions_usd"] <= 500, "export_category"] = "low"` |
| Label exports above 500 | `my_df.loc[my_df["export_milions_usd"] > 500, "export_category"] = "high"` |
| Calculate mean exports by country | `my_df.groupby("country")["export_milions_usd"].mean()` |
| Check an object's type | `type(my_df)` |
| Create an output folder if needed | `output_dir.mkdir(exist_ok=True)` |
| Save to Excel without row labels | `my_df.to_excel(output_path, index=False)` |
| List items in a folder | `tables_dir.iterdir()` |
| Get a filename as text | `file.name` |
| Check a filename's ending | `file.name.endswith(".xlsx")` |
| Add a filename to a list | `excel_files.append(file.name)` |